# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Metadata as an object, print summary info
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Refer to all items using their `@id` fields as required by Croissant metadata. We'll print metadata for record sets and show available fields per set.

In [ ]:
# List all record sets by their `@id`
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the metadata.")
else:
    for rs in record_sets:
        print(f"Record set: {rs['@id']}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        print(f"  Description: {rs.get('description', 'N/A')}")

        if 'field' in rs:
            print("  Fields:")
            for field in rs['field']:
                if isinstance(field, dict):
                    print(f"    {field['@id']} : {field.get('name', 'N/A')}")
                elif isinstance(field, str):
                    print(f"    {field}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, we'll use the first record set if available
dataframes = {}
if not record_sets:
    print("No record sets available to extract data.")
else:
    # Use the first record set
    first_rs = record_sets[0]['@id']
    print(f"Extracting data from record set @id: {first_rs}")
    records = list(dataset.records(record_set=first_rs))
    if records:
        df = pd.DataFrame(records)
        dataframes[first_rs] = df
        print("Fields in this record set:")
        print(df.columns.tolist())
        display(df.head())
    else:
        print('No records found in this record set.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming data distributions, and grouping data. All references to fields use their `@id` names as found in the DataFrame.

In [ ]:
# Pick a numeric field (by @id) for demonstration
if not dataframes:
    print('No DataFrame available for EDA.')
else:
    sample_df = next(iter(dataframes.values()))
    # List numeric columns
    numeric_cols = sample_df.select_dtypes(include=[int, float]).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field '@id': {numeric_field_id}")
        # Set a threshold for demonstration
        threshold = sample_df[numeric_field_id].mean() if not pd.isnull(sample_df[numeric_field_id].mean()) else 0
        filtered_df = sample_df[sample_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Group by another field if any string columns
        group_col_candidates = sample_df.select_dtypes(include=[object]).columns.tolist()
        group_field = None
        if group_col_candidates:
            for col in group_col_candidates:
                if sample_df[col].nunique() < 10:
                    group_field = col
                    break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
            print(f"Grouped average of {numeric_field_id} by {group_field}:")
            display(grouped_df.head())
    else:
        print('No numeric field found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_cols:
    df = next(iter(dataframes.values()))
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    if group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Dataset and record sets loaded via Croissant schema using `mlcroissant`.
- Data can be programmatically extracted by record set and referenced by `@id` for both record sets and fields.
- Basic EDA and visualization demonstrated on available numeric fields.

> For further analysis, refer to the dataset documentation and field definitions in the Croissant schema for precise interpretation of each field (`@id`).